# Audit labels and build task datasets

See README.md for prerequisites. Restart the kernel and run this notebook from the repository root.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import os, re, pickle
from pathlib import Path

In [ ]:
# ── Load all four source files ──
bank_train = pd.read_csv("data/processed/banking77_train_clean.csv")
bank_test  = pd.read_csv("data/processed/banking77_test_clean.csv")
phrasebank = pd.read_csv("data/processed/financial_phrasebank_clean.csv")
LABELED_PATH = Path("data/labeled/labeled_reviews_v3.csv")
if not LABELED_PATH.is_file():
    raise FileNotFoundError("Run updated notebook 4 and manually audit the v3 labels before rebuilding training data.")
scraped = pd.read_csv(LABELED_PATH)
if "taxonomy_version" not in scraped or not scraped["taxonomy_version"].eq("review-taxonomy-v3").all():
    raise ValueError("Mixed/old taxonomy labels; audit and relabel before training")

print(f"Banking77 train : {bank_train.shape}   columns: {list(bank_train.columns)}")
print(f"Banking77 test  : {bank_test.shape}   columns: {list(bank_test.columns)}")
print(f"PhraseBank      : {phrasebank.shape}   columns: {list(phrasebank.columns)}")
print(f"Scraped reviews : {scraped.shape}   columns: {list(scraped.columns)}")
TAXONOMY_SHA256 = __import__("hashlib").sha256(Path("taxonomy.json").read_bytes()).hexdigest()
if "taxonomy_sha256" not in scraped or not scraped["taxonomy_sha256"].eq(TAXONOMY_SHA256).all():
    raise ValueError("Label file belongs to a different taxonomy; review/relabel first")


## Step 1 · Map Banking77 77 Intents → CRO Business Taxonomy

In [ ]:
# ── Banking77 classes not relevant to mobile banking / digital wallets ──
IRRELEVANT_B77 = {
    "card_about_to_expire", "card_arrival", "card_delivery_estimate",
    "card_swallowed", "get_physical_card", "getting_spare_card",
    "order_physical_card", "card_linking",
    "atm_support", "cash_withdrawal_charge", 
    "declined_cash_withdrawal", "pending_cash_withdrawal",
    "wrong_amount_of_cash_received", "top_up_by_cash_or_cheque",
    "balance_not_updated_after_cheque_or_cash_deposit",
    
    "disposable_card_limits", "get_disposable_virtual_card", "getting_virtual_card",
    "visa_or_mastercard", "supported_cards_and_currencies",
    "fiat_currency_support", "country_support", "age_limit",
    "apple_pay_or_google_pay",
    "why_is_my_card_in_a_cooldown_period",
    
}

print(f"Irrelevant classes to drop: {len(IRRELEVANT_B77)}")

In [ ]:
# ── Deterministic mapping: Banking77 intent → (CRO intent, CRO root cause) ──
CLEAN_MAP = {
    # Data / balance issues
    "balance_not_updated_after_bank_transfer": ("data_integrity_issue", "data_sync_issue"),
    "transfer_into_account": ("data_integrity_issue", "data_sync_issue"),
    "refund_not_showing_up": ("data_integrity_issue", "data_sync_issue"),

    # Payment failures
    "failed_transfer": ("payment_failure", "api_failure"),
    "pending_transfer": ("payment_failure", "api_failure"),
    "declined_transfer": ("payment_failure", "api_failure"),
    "cancel_transfer": ("payment_failure", "api_failure"),
    "top_up_failed": ("payment_failure", "api_failure"),
    "top_up_reverted": ("payment_failure", "api_failure"),
    "pending_top_up": ("payment_failure", "api_failure"),

    # Interbank / LankaPay
    "transfer_not_received_by_recipient": ("interbank_transfer_failure", "lankapay_ceft_failure"),

    # Refund / billing
    "request_refund": ("refund_request", "unjustified_billing"),
    "extra_charge_on_statement": ("refund_request", "unjustified_billing"),
    "transfer_fee_charged": ("refund_request", "fee_without_service"),
    "card_payment_fee_charged": ("refund_request", "fee_without_service"),

    # FX / remittance
    "exchange_rate": ("fx_conversion_complaint", "fx_rate_dispute"),
    "exchange_charge": ("fx_conversion_complaint", "fx_rate_dispute"),
    "exchange_via_app": ("fx_conversion_complaint", "fx_rate_dispute"),
    "card_payment_wrong_exchange_rate": ("fx_conversion_complaint", "fx_rate_dispute"),
    "wrong_exchange_rate_for_transfer": ("fx_conversion_complaint", "fx_rate_dispute"),

    # Access / auth
    "passcode_forgotten": ("access_blocked", "authentication_failure"),
    "pin_blocked": ("access_blocked", "authentication_failure"),
    "beneficiary_not_allowed": ("access_blocked", "device_limit_exceeded"),
    "activate_my_card": ("access_blocked", "authentication_failure"),

    # Regulatory / KYC
    "verify_my_identity": ("access_blocked", "regulatory_block"),
    "unable_to_verify_identity": ("access_blocked", "regulatory_block"),
    "verify_source_of_funds": ("regulatory_risk", "regulatory_block"),
    "verify_top_up": ("regulatory_risk", "regulatory_block"),
    "identity_document_queried": ("regulatory_risk", "regulatory_block"),

    # Security
    "lost_or_stolen_card": ("security_concern", "poor_security_implementation"),
    "compromised_card": ("security_concern", "poor_security_implementation"),

    # Technical / app broken
    "virtual_card_not_working": ("technical_failure", "app_crash_or_bug"),
    "contactless_not_working": ("technical_failure", "app_crash_or_bug"),

    # UX / friction
    "top_up_limits": ("ux_friction", "offer_discoverability"),
    "top_up_by_bank_transfer_only": ("ux_friction", "offer_discoverability"),
    "top_up_by_card_charge": ("ux_friction", "offer_discoverability"),
    "change_pin": ("ux_friction", "offer_discoverability"),
    "why_verify_identity": ("ux_friction", "offer_discoverability"),

    # Feature requests
    "automatic_top_up": ("feature_request", "offer_discoverability"),
    "edit_personal_details": ("feature_request", "offer_discoverability"),
    "receiving_money": ("feature_request", "offer_discoverability"),

    # Service quality
    "transfer_timing": ("service_quality", "slow_performance"),

    # Account closure
    "terminate_account": ("account_closure", "unresponsive_support")
}
# Unrecognized financial activity is suspected fraud, not an ordinary refund.
for category in ("card_payment_not_recognised", "cash_withdrawal_not_recognised", "direct_debit_payment_not_recognised"):
    CLEAN_MAP[category] = ("security_concern", "suspected_unauthorized_transaction")
for category in ("compromised_card", "identity_document_stolen", "lost_or_stolen_phone", "lost_or_stolen_card"):
    CLEAN_MAP[category] = ("security_concern", "other")
# Banking77 category alone cannot prove a root cause or a Sri Lankan payment rail.
CLEAN_MAP["terminate_account"] = ("account_closure", "other")
CLEAN_MAP["transfer_not_received_by_recipient"] = ("payment_failure", "other")


In [ ]:
# ── Combine Banking77 train + test ──
bank_all = pd.concat([bank_train, bank_test], ignore_index=True)
print(f"Banking77 combined: {len(bank_all)} rows")

# Normalize the label_text column
bank_all["b77_intent"] = (bank_all["label_text"]
    .astype(str).str.strip().str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Drop irrelevant classes
before = len(bank_all)
bank_all = bank_all[~bank_all["b77_intent"].isin(IRRELEVANT_B77)].copy()
print(f"Dropped {before - len(bank_all)} irrelevant rows -> {len(bank_all)} remain")

# Map each row to CRO intent and root cause
cro_intents = []
cro_root_causes = []

for i, row in bank_all.iterrows():
    b77 = row["b77_intent"]
    text = str(row["text"]).lower()

    if b77 in CLEAN_MAP:
        intent, root = CLEAN_MAP[b77]

    # ── Ambiguous intents resolved by keywords ──
    elif b77 == "card_not_working":
        if re.search(r"block|suspend|frozen|access|login|open", text):
            intent, root = "access_blocked", "authentication_failure"
        else:
            intent, root = "technical_failure", "app_crash_or_bug"

    elif b77 == "card_acceptance":
        if re.search(r"declin|reject|fail|merchant|shop|store|not accept", text):
            intent, root = "payment_failure", "api_failure"
        else:
            intent, root = "ux_friction", "offer_discoverability"

    elif b77 == "card_payment_not_recognised":
        if re.search(r"refund|money back|charge|reverse|didn.t make|unauthori", text):
            intent, root = "refund_request", "unjustified_billing"
        else:
            intent, root = "data_integrity_issue", "data_sync_issue"

    elif b77 == "pending_card_payment":
        if re.search(r"day|week|long|still|never|wrong", text):
            intent, root = "data_integrity_issue", "data_sync_issue"
        else:
            intent, root = "payment_failure", "api_failure"

    elif b77 == "declined_card_payment":
        if re.search(r"block|suspend|fraud|unauthori|security", text):
            intent, root = "access_blocked", "regulatory_block"
        else:
            intent, root = "payment_failure", "api_failure"

    elif b77 == "reverted_card_payment?":
        if re.search(r"refund|want|please|return|back", text):
            intent, root = "refund_request", "unjustified_billing"
        else:
            intent, root = "data_integrity_issue", "data_sync_issue"

    else:
        # Unmapped intent -> noise
        intent, root = "noise", "unreadable_text"

    cro_intents.append(intent)
    cro_root_causes.append(root)

bank_all["cro_intent"] = cro_intents
bank_all["cro_root_cause"] = cro_root_causes

print(bank_all["cro_intent"].value_counts().to_string())

In [ ]:
# ── Save mapped Banking77 ──
bank_all[["text", "b77_intent", "cro_intent", "cro_root_cause"]].to_csv(
    "data/processed/banking77_mapped.csv", index=False
)
print(f"({len(bank_all)})")

## Step 2 · Scraped Review Quality Audit

In [ ]:
# ── Scraped review summary ──
print(f"Scraped reviews: {scraped.shape}")
print(f"Text column: 'review_text'")
print(f"Unique sentiments: {scraped['sentiment'].nunique()}")
print(f"Unique intents: {scraped['intent'].nunique()}")
print(f"Unique root causes: {scraped['root_cause'].nunique()}")
print(f"Rows with risk_score: {scraped['risk_score'].notna().sum()}")

In [ ]:
# ── Label value counts ──
for col_name in ["sentiment", "intent", "root_cause", "risk_band"]:
    print(f"SCRAPED — {col_name.upper()}")
    if col_name in scraped.columns:
        print(scraped[col_name].value_counts())
    else:
        print(f"  Column '{col_name}' not found")
    print()

In [ ]:
# ── Null / blank / placeholder check ──
label_cols = ["sentiment", "intent", "root_cause"]
bad_values = ["unknown", "Unknown", "UNKNOWN", "", "nan", "none", "None", "NaN"]

total_bad = 0
for col in label_cols:
    n_null = scraped[col].isna().sum()
    n_bad  = scraped[col].isin(bad_values).sum()
    total_bad += n_null + n_bad
    status = "OK" if (n_null + n_bad) == 0 else "ISSUE"
    print(f"  {status}  {col:15s}  null={n_null}  bad_string={n_bad}")

print()
if total_bad == 0:
    print("No unknown or blank labels found")
else:
    print(f"Total bad labels: {total_bad} — review needed")

In [ ]:
# ── Spot-check noise, other, and low-confidence rows ──
noise_df = scraped[scraped["intent"] == "noise"]
print(f"intent == 'noise': {len(noise_df)} rows")
if len(noise_df) > 0:
    display(noise_df[["review_text", "sentiment", "root_cause", "risk_score"]].head(10))

print()
other_rc = scraped[scraped["root_cause"] == "other"]
print(f"root_cause == 'other': {len(other_rc)} rows")
if len(other_rc) > 0:
    display(other_rc[["review_text", "sentiment", "intent", "risk_score"]].head(10))

print()
if "confidence" in scraped.columns:
    low_conf = scraped[scraped["confidence"] < 0.6]
    print(f"confidence < 0.6: {len(low_conf)} rows (manual review recommended)")
    if len(low_conf) > 0:
        display(low_conf[["review_text", "intent", "root_cause", "confidence"]].head(10))

In [ ]:
# Filter invalid labeler outputs; filter task-specific noise separately below.
if "confidence" in scraped:
    scraped["confidence"] = pd.to_numeric(scraped["confidence"], errors="coerce")
    scraped = scraped[scraped["confidence"].ge(0.6)].copy()
taxonomy = __import__("json").loads(Path("taxonomy.json").read_text(encoding="utf-8"))
for column, allowed in [("sentiment", taxonomy["sentiments"]), ("intent", taxonomy["intents"]), ("root_cause", taxonomy["root_causes"])]:
    scraped = scraped[scraped[column].isin(allowed)].copy()
print("Rows after schema/confidence checks:", len(scraped))
print("IMPORTANT: old labels are not automatically corrected by a taxonomy edit. Audit/relabel before final training.")


In [ ]:
# ── High-risk complaints ──
high_risk = scraped[(scraped["intent"] == "complaint") & (scraped["risk_score"] >= 7)]
print(f"complaint + risk_score >= 7: {len(high_risk)} rows")
if len(high_risk) > 0:
    display(high_risk[["review_text", "sentiment", "root_cause", "risk_score", "risk_band"]].head(10))

## Step 3 · Build Four Datasets

In [ ]:
# ── Build Sentiment Dataset (PhraseBank + Scraped) ──

# PhraseBank: columns are ['label', 'text']
pb_sent = phrasebank[["text", "label"]].copy()
pb_sent.columns = ["text", "sentiment"]
pb_sent["sentiment"] = pb_sent["sentiment"].str.lower().str.strip()

# Scraped: text column is 'review_text'
sc_sent = scraped[["review_text", "sentiment"]].copy()
sc_sent.columns = ["text", "sentiment"]

# Combine, clean, deduplicate
sentiment_df = pd.concat([pb_sent, sc_sent], ignore_index=True)
sentiment_df = sentiment_df.dropna(subset=["text", "sentiment"])
sentiment_df["text"] = sentiment_df["text"].str.strip()
sentiment_df = sentiment_df[sentiment_df["text"].str.len() > 3]

print(f"Sentiment dataset: {sentiment_df.shape}")
print(sentiment_df["sentiment"].value_counts())


In [ ]:
# ── Build Intent Dataset (MAPPED Banking77 + Scraped) ──
# IMPORTANT: Use the CRO-mapped Banking77 intents, NOT the raw 77 classes

# Banking77: use cro_intent column from the mapping step above
b77_int = bank_all[["text", "cro_intent"]].copy()
b77_int.columns = ["text", "intent"]

# Scraped: text column is 'review_text'
sc_int = scraped[["review_text", "intent"]].copy()
sc_int.columns = ["text", "intent"]

# Combine, clean, deduplicate
intent_df = pd.concat([b77_int, sc_int], ignore_index=True)
intent_df = intent_df.dropna(subset=["text", "intent"])
intent_df["text"] = intent_df["text"].str.strip()
intent_df = intent_df[intent_df["text"].str.len() > 3]

print(f"Intent dataset: {intent_df.shape}")
print(f"Number of classes: {intent_df['intent'].nunique()}")
print(intent_df["intent"].value_counts())
intent_df = intent_df[intent_df["intent"] != "noise"].copy()


In [ ]:
# ── Build Root Cause Dataset (Scraped only) ──
rc_df = scraped[["review_text", "root_cause"]].copy()
rc_df.columns = ["text", "root_cause"]
rc_df = rc_df.dropna(subset=["text", "root_cause"])
rc_df["text"] = rc_df["text"].str.strip()
rc_df = rc_df[rc_df["text"].str.len() > 3]

print(f"Root cause dataset: {rc_df.shape}")
print(rc_df["root_cause"].value_counts())
rc_df = rc_df[~rc_df["root_cause"].isin(["other", "unreadable_text"])].copy()


In [ ]:
# ── Build Risk Dataset (Scraped only) ──
risk_df = scraped[["review_text", "risk_score", "risk_band"]].copy()
risk_df.columns = ["text", "risk_score", "risk_band"]
risk_df = risk_df.dropna(subset=["text", "risk_score"])
risk_df["text"] = risk_df["text"].str.strip()
risk_df["risk_score"] = pd.to_numeric(risk_df["risk_score"], errors="coerce")
risk_df = risk_df.dropna(subset=["risk_score"])
risk_df = risk_df[risk_df["text"].str.len() > 3]

# Encode risk_band into risk_label integer
risk_band_order = ["NEGLIGIBLE", "LOW", "MODERATE", "HIGH", "CRITICAL"]
risk_df["risk_band"] = risk_df["risk_band"].str.upper().str.strip()
risk_df["risk_label"] = pd.Categorical(
    risk_df["risk_band"], categories=risk_band_order, ordered=True
).codes.astype(int)

if (risk_df["risk_label"] < 0).any():
    print("WARNING: Unknown risk_band values found!")
    print(risk_df[risk_df["risk_label"] < 0]["risk_band"].unique())
else:
    print("All risk_band values are valid")

print(f"\nRisk dataset: {risk_df.shape}")
print(f"Columns: {list(risk_df.columns)}")
print(risk_df["risk_band"].value_counts())

# Recompute bands from validated scores instead of trusting inconsistent old strings.
risk_df = risk_df[risk_df["risk_score"].between(0, 10)].copy()
risk_df["risk_band"] = pd.cut(risk_df["risk_score"], [-0.001, 2, 4, 6, 8, 10], labels=risk_band_order).astype(str)
risk_df["risk_label"] = risk_df["risk_band"].map({name:i for i,name in enumerate(risk_band_order)})


In [ ]:
# Consistent exclusions BEFORE label encoders are fitted.
from training_quality import clean_task
quality_dir = Path("reports/training_quality")
quality_dir.mkdir(parents=True, exist_ok=True)
for task_name, frame, target in [("sentiment", sentiment_df, "sentiment"), ("intent", intent_df, "intent"), ("root_cause", rc_df, "root_cause"), ("risk", risk_df, "risk_band")]:
    cleaned, excluded = clean_task(frame, target, min_count=10)
    excluded.to_csv(quality_dir / f"{task_name}_excluded.csv", index=False)
    if task_name == "sentiment": sentiment_df = cleaned
    elif task_name == "intent": intent_df = cleaned
    elif task_name == "root_cause": rc_df = cleaned
    else: risk_df = cleaned
    print(task_name, "excluded:", excluded["exclusion_reason"].value_counts().to_dict())

# ── Save all four datasets ──
OUT = "data/model_ready"
Path(OUT).mkdir(parents=True, exist_ok=True)

sentiment_df.to_csv(f"{OUT}/sentiment_dataset.csv", index=False)
intent_df.to_csv(f"{OUT}/intent_dataset.csv", index=False)
rc_df.to_csv(f"{OUT}/root_cause_dataset.csv", index=False)
risk_df.to_csv(f"{OUT}/risk_dataset.csv", index=False)

print(f"sentiment_dataset.csv  : {sentiment_df.shape}")
print(f"intent_dataset.csv     : {intent_df.shape}")
print(f"root_cause_dataset.csv : {rc_df.shape}")
print(f"risk_dataset.csv       : {risk_df.shape}")
print(f"\nAll saved to {OUT}/")

## Step 4 · Label Encoding

In [ ]:
# ── Sentiment label encoding ──
le_sentiment = LabelEncoder()
sentiment_df["sentiment_label"] = le_sentiment.fit_transform(sentiment_df["sentiment"])

print("Sentiment encoding:")
for idx, label in enumerate(le_sentiment.classes_):
    count = (sentiment_df["sentiment"] == label).sum()
    print(f"  {idx}  <-  {label:12s}  ({count} rows)")

In [ ]:
# ── Intent label encoding ──
le_intent = LabelEncoder()
intent_df["intent_label"] = le_intent.fit_transform(intent_df["intent"])

print("Intent encoding:")
for idx, label in enumerate(le_intent.classes_):
    count = (intent_df["intent"] == label).sum()
    print(f"  {idx:2d}  <-  {label:30s}  ({count} rows)")

In [ ]:
# ── Root cause label encoding ──
le_root_cause = LabelEncoder()
rc_df["root_cause_label"] = le_root_cause.fit_transform(rc_df["root_cause"])

print("Root cause encoding:")
for idx, label in enumerate(le_root_cause.classes_):
    count = (rc_df["root_cause"] == label).sum()
    print(f"  {idx:2d}  <-  {label:35s}  ({count} rows)")

In [ ]:
# ── Save label encoders and encoded CSVs ──
enc_dir = f"{OUT}/encoders"
Path(enc_dir).mkdir(exist_ok=True)

for name, enc in [("sentiment", le_sentiment), ("intent", le_intent), ("root_cause", le_root_cause)]:
    with open(f"{enc_dir}/le_{name}.pkl", "wb") as f:
        pickle.dump(enc, f)
    print(f"  le_{name}.pkl saved ({len(enc.classes_)} classes)")

sentiment_df.to_csv(f"{OUT}/sentiment_encoded.csv", index=False)
intent_df.to_csv(f"{OUT}/intent_encoded.csv", index=False)
rc_df.to_csv(f"{OUT}/root_cause_encoded.csv", index=False)

print("\nEncoded CSVs saved to", OUT)
# Keep provenance with regenerated data so tokenization cannot silently use older labels.
import json, hashlib
dataset_manifest = {"taxonomy_sha256": TAXONOMY_SHA256,
    "labeled_source": str(LABELED_PATH), "labeled_source_sha256": hashlib.sha256(LABELED_PATH.read_bytes()).hexdigest(),
    "files": {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in Path(OUT).glob("*_dataset.csv")}}
(Path(OUT) / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")


## Step 5 · Class Imbalance Analysis

In [ ]:
# ── Sentiment imbalance ──
vc = sentiment_df["sentiment"].value_counts()
pct = (vc / vc.sum() * 100).round(1)
cols = {"positive": "#4CAF50", "negative": "#F44336", "neutral": "#9E9E9E", "mixed": "#FF9800"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Sentiment — Class Distribution", fontweight="bold")

ax = axes[0]
bars = ax.bar(vc.index, vc.values, color=[cols.get(c, "#888") for c in vc.index], edgecolor="white")
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f"{val:,}", ha="center", fontsize=9)
ax.set_title("Count per Class")
ax.set_ylabel("Count")

ax = axes[1]
ax.pie(vc.values, labels=[f"{l}\n{p}%" for l, p in zip(vc.index, pct)],
       colors=[cols.get(c, "#888") for c in vc.index], startangle=90)
ax.set_title("% Share")

plt.tight_layout()
plt.savefig(f"{OUT}/imbalance_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

ratio = vc.max() / vc.min()
print(f"Imbalance ratio: {ratio:.1f}x")
if ratio > 5:
    print("Severe imbalance — use class weights")

In [ ]:
# ── Intent imbalance ──
vc2 = intent_df["intent"].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
threshold = vc2.max() * 0.05
bar_cols = ["#F44336" if v < threshold else "#5C85D6" for v in vc2.values]
ax.barh(vc2.index, vc2.values, color=bar_cols, edgecolor="white")
ax.axvline(threshold, color="red", linestyle="--", lw=1.2, label=f"5% threshold ({int(threshold)} rows)")
ax.set_title("Intent — Class Distribution (red = minority)", fontweight="bold")
ax.set_xlabel("Count")
ax.legend(fontsize=8)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f"{OUT}/imbalance_intent.png", dpi=150, bbox_inches="tight")
plt.show()

ratio2 = vc2.max() / vc2.min()
print(f"Imbalance ratio: {ratio2:.1f}x")
print(f"Classes below 5% threshold: {(vc2 < threshold).sum()}")

In [ ]:
# ── Root cause imbalance ──
vc3 = rc_df["root_cause"].value_counts()

fig, ax = plt.subplots(figsize=(12, 8))
threshold3 = vc3.max() * 0.05
bar_cols3 = ["#F44336" if v < threshold3 else "#9B59B6" for v in vc3.values]
ax.barh(vc3.index, vc3.values, color=bar_cols3, edgecolor="white")
ax.axvline(threshold3, color="red", linestyle="--", lw=1.2, label=f"5% threshold ({int(threshold3)} rows)")
ax.set_title("Root Cause — Class Distribution (red = minority)", fontweight="bold")
ax.set_xlabel("Count")
ax.legend(fontsize=8)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f"{OUT}/imbalance_root_cause.png", dpi=150, bbox_inches="tight")
plt.show()

ratio3 = vc3.max() / vc3.min()
print(f"Imbalance ratio: {ratio3:.1f}x")
print(f"Classes with < 50 samples: {(vc3 < 50).sum()}")
if (vc3 < 50).sum() > 0:
    print(vc3[vc3 < 50].to_string())

In [ ]:
# ── Risk band imbalance ──
band_order = ["NEGLIGIBLE", "LOW", "MODERATE", "HIGH", "CRITICAL"]
band_cols = {"NEGLIGIBLE": "#e8e8e8", "LOW": "#b3e6b3", "MODERATE": "#ffd04d",
             "HIGH": "#ff944d", "CRITICAL": "#ff4d4d"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Risk — Score and Band Distribution", fontweight="bold")

ax = axes[0]
ax.hist(risk_df["risk_score"], bins=range(0, 12), color="#9B59B6", edgecolor="white", rwidth=0.8)
ax.axvline(7, color="orange", linestyle="--", lw=1.5, label="High threshold (7)")
ax.axvline(9, color="red", linestyle="--", lw=1.5, label="Critical threshold (9)")
ax.set_title("Risk Score Distribution (0-10)")
ax.set_xlabel("Risk Score")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

ax = axes[1]
vc4 = risk_df["risk_band"].value_counts().reindex(band_order, fill_value=0)
ax.bar(vc4.index, vc4.values, color=[band_cols.get(b, "#888") for b in vc4.index], edgecolor="white")
for i, (k, v) in enumerate(vc4.items()):
    ax.text(i, v + 5, str(v), ha="center", fontsize=9)
ax.set_title("Risk Band Distribution")

plt.tight_layout()
plt.savefig(f"{OUT}/imbalance_risk.png", dpi=150, bbox_inches="tight")
plt.show()

print("Risk band distribution:")
for band, cnt in vc4.items():
    pct_val = cnt / vc4.sum() * 100
    print(f"  {band:12s} : {cnt:5d}  ({pct_val:.1f}%)")

In [ ]:
# ── Final Summary ──
print(f"Sentiment  : {sentiment_df.shape[0]:>6,} rows, {sentiment_df['sentiment'].nunique()} classes")
print(f"Intent     : {intent_df.shape[0]:>6,} rows, {intent_df['intent'].nunique()} classes")
print(f"Root Cause : {rc_df.shape[0]:>6,} rows, {rc_df['root_cause'].nunique()} classes")
print(f"Risk       : {risk_df.shape[0]:>6,} rows, {risk_df['risk_band'].nunique()} bands")
